In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
#from langchain.document_loaders import TextLoader
#from langchain.schema.document import Document
import textwrap
import chromadb
from rich.console import Console
from rich.markdown import Markdown

Load the desired pdf file

In [6]:
import fitz  # PyMuPDF

# This is the exact equivalent of your 3 lines
doc = fitz.open("ACADEMIC_POLICY_MANUAL_14v6.pdf")
pages = [page.get_text() for page in doc]
doc.close()

print(f"Loaded {len(pages)} pages")

Loaded 66 pages


In [7]:
'''
from sentence_transformers import SentenceTransformer
# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
# The queries and documents to embed
queries = [
    "What is the capital of China?",
    "Explain gravity",
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings, document_embeddings)
print(similarity)
prit(len(query_embeddings[0]))
'''

'\nfrom sentence_transformers import SentenceTransformer\n# Load the model\nmodel = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")\n# The queries and documents to embed\nqueries = [\n    "What is the capital of China?",\n    "Explain gravity",\n]\ndocuments = [\n    "The capital of China is Beijing.",\n    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",\n]\nquery_embeddings = model.encode(queries, prompt_name="query")\ndocument_embeddings = model.encode(documents)\n\n# Compute the (cosine) similarity between the query and document embeddings\nsimilarity = model.similarity(query_embeddings, document_embeddings)\nprint(similarity)\nprit(len(query_embeddings[0]))\n'

Load Embedding Model {Qwen or BAAI or miniLLM}

In [8]:

embeddings = HuggingFaceEmbeddings(
    #model_name="BAAI/bge-small-en-v1.5",  # Small, fast, excellent performance
    model_name="Qwen/Qwen3-Embedding-0.6B",  # Larger, more accurate, but slower
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Setting up permanent vector store - Run it once or as needed

In [9]:
'''


# Create client with explicit settings
client = chromadb.PersistentClient(path='chroma/IBA_AM_2026/pages')

vectordb_pages = Chroma(
    client=client,
    collection_name="page_chunks",
    embedding_function=embeddings
)

# Add your texts
vectordb_pages.add_texts(texts=pages)
print(f"Page-level chunks: {vectordb_pages._collection.count()}")
'''

'\n\n\n# Create client with explicit settings\nclient = chromadb.PersistentClient(path=\'chroma/IBA_AM_2026/pages\')\n\nvectordb_pages = Chroma(\n    client=client,\n    collection_name="page_chunks",\n    embedding_function=embeddings\n)\n\n# Add your texts\nvectordb_pages.add_texts(texts=pages)\nprint(f"Page-level chunks: {vectordb_pages._collection.count()}")\n'

Loading an already stored embeddings

In [10]:
# Load page-level chunks
client_pages = chromadb.PersistentClient(path='chroma/IBA_AM_2026/pages')
vectordb_pages = Chroma(
    client=client_pages,
    collection_name="page_chunks",
    embedding_function=embeddings
)
print(f"Pages: {vectordb_pages._collection.count()} chunks")

Pages: 66 chunks


/tmp/ipykernel_1151/3138023292.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb_pages = Chroma(


Making Embedding of the Query Question

In [11]:
question = "What is the makeup exam policy for health related reasons?"
#compute embeddings for the question
question_vector = embeddings.embed_query(question)
docs_pages = vectordb_pages.similarity_search_with_score(question,k=3)
#docs2 = vectordb.max_marginal_relevance_search(question,k=3)

In [12]:
docs_pages

[(Document(metadata={}, page_content="Academic Policy Manual \n \n2014-2015 \nIBA Academic sub-committee/ Version Number: 0006 \nMake-Up Examinations \nI. For Morning Program Students of both Campuses. \nNo make-up examination shall be allowed for missing Term or Semester Final \nExamination under normal circumstances. \nII. For Evening Program Students. \nEvening program students, who are sent out of Karachi during term and final \nexaminations on official assignments by their respective organizations, may be \nallowed to take make-up examinations under following conditions: \na) This facility will be allowed to the Evening Program students only for one of the two \nterm examinations for the courses taken by them. \nb) This facility shall also be allowed for the semester final exams if the student has not \nalready availed this facility for the term examinations. The concerned student shall \nbe required to provide the following documents at least one week before the \nscheduled exam:

In [13]:
for result in docs_pages:
    print("\n")
    print(result[1])
    print(result[0].page_content)



0.7608117461204529
Academic Policy Manual 
 
2014-2015 
IBA Academic sub-committee/ Version Number: 0006 
Make-Up Examinations 
I. For Morning Program Students of both Campuses. 
No make-up examination shall be allowed for missing Term or Semester Final 
Examination under normal circumstances. 
II. For Evening Program Students. 
Evening program students, who are sent out of Karachi during term and final 
examinations on official assignments by their respective organizations, may be 
allowed to take make-up examinations under following conditions: 
a) This facility will be allowed to the Evening Program students only for one of the two 
term examinations for the courses taken by them. 
b) This facility shall also be allowed for the semester final exams if the student has not 
already availed this facility for the term examinations. The concerned student shall 
be required to provide the following documents at least one week before the 
scheduled exam: 
a. A certificate from his/her or

In [14]:
print(docs_pages[0][0].page_content)

Academic Policy Manual 
 
2014-2015 
IBA Academic sub-committee/ Version Number: 0006 
Make-Up Examinations 
I. For Morning Program Students of both Campuses. 
No make-up examination shall be allowed for missing Term or Semester Final 
Examination under normal circumstances. 
II. For Evening Program Students. 
Evening program students, who are sent out of Karachi during term and final 
examinations on official assignments by their respective organizations, may be 
allowed to take make-up examinations under following conditions: 
a) This facility will be allowed to the Evening Program students only for one of the two 
term examinations for the courses taken by them. 
b) This facility shall also be allowed for the semester final exams if the student has not 
already availed this facility for the term examinations. The concerned student shall 
be required to provide the following documents at least one week before the 
scheduled exam: 
a. A certificate from his/her organization giving det

Connecting with LM Studio

In [15]:
# Example: reuse your existing OpenAI setup
from langchain_openai import ChatOpenAI

# Point to the local server
llm = ChatOpenAI(
    base_url="http://172.19.64.1:1234/v1",
    api_key="lm-studio",
    #model="google/gemma-3-12b",
    model="tiny-aya-global",
    temperature=0.7
)

Calling LM Studio model with the given prompt

In [16]:
response1 = docs_pages[0][0].page_content
response2 = docs_pages[1][0].page_content
response3 = docs_pages[2][0].page_content

prompt = f"""
Based on the following information:\n\n
1. {response1}\n\n
2. {response2}\n\n
3. {response3}\n\n
Please provide a detailed answer to the question: {question}.
Your answer should integrate the essence of all three responses, providing a unified answer that leverages the \
diverse perspectives or data points provided by three responses. \
If the responses are irrelevant to the question then respond by saying that I couldn't find a good response to your query in the database. 
"""

response = llm.invoke(prompt)
console = Console()

md = Markdown(response.content)
console.print(md)

The makeup exam policy for health-related reasons at the Institute of Business Administration (IBA) is outlined in 
the 2014-2015 Academic Policy Manual and includes several conditions:                                              

                                                1. Medical Grounds                                                 

 • Allowed Only in Serious Cases: A make-up examination may be permitted on medical grounds only in extremely      
   serious cases authenticated by recognized hospitals.                                                            
 • Limited to One Term Exam per Semester: No make-up of a semester final exam is allowed due to health reasons,    
   regardless of the type of illness or condition.                                                                 
 • Decision-Making Authority: The Executive Committee makes the final decision on such requests and ensures that   
   they are based on valid medical documentation.                                                                  

                                           2. Spouse or Parent’s Illness                                           

 • Expanded Policy: The policy now covers cases where a student’s spouse is hospitalized in critical condition or  
   when the student’s parent passes away (as per the 57th Meeting of the Academic Board).                          
 • Evidence Requirement: Students must provide documentary evidence to substantiate their request.                 

                                               3. General Conditions                                               

 • One Term Exam Exception: A make-up exam is only allowed for one of the two term examinations in a semester, not 
   both.                                                                                                           
 • Exam Date Constraints: The student must appear within three weeks of the original exam date for a term exam and 
   within six weeks for a semester final exam.                                                                     
 • Additional Fee: A fee of Rs. 2000 is required for making up an exam.                                            

                                                      Summary                                                      

The makeup policy for health-related reasons at IBA is strict, limited to specific scenarios (medical emergencies  
or family crises), and allows only one make-up per term. Students must meet deadlines and pay a fee. The policy    
emphasizes authenticity of documentation and final decisions made by the Executive Committee.                      

For further clarification, refer directly to the IBA Academic Policy Manual 2014-2015, as this is the primary      
source for these regulations.<|END_RESPONSE|>

In [17]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)
from langchain_experimental.text_splitter import SemanticChunker

Other Strategies: Setting up permanent vector store - Run it once or as needed

In [18]:
'''
# Combine all pages into one text for chunking
full_text = "\n\n".join(pages)

# ===== Strategy 2: Fixed-size chunks =====
fixed_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=50
)
fixed_chunks = fixed_splitter.split_text(full_text)

client_fixed = chromadb.PersistentClient(path='chroma/IBA_AM_2026/fixed')
vectordb_fixed = Chroma(
    client=client_fixed,
    collection_name="fixed_chunks",
    embedding_function=embeddings
)
vectordb_fixed.add_texts(texts=fixed_chunks)
print(f"Fixed chunks: {vectordb_fixed._collection.count()}")

# ===== Strategy 3: Recursive/Paragraph chunking =====
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
recursive_chunks = recursive_splitter.split_text(full_text)

client_recursive = chromadb.PersistentClient(path='chroma/IBA_AM_2026/recursive')
vectordb_recursive = Chroma(
    client=client_recursive,
    collection_name="recursive_chunks",
    embedding_function=embeddings
)
vectordb_recursive.add_texts(texts=recursive_chunks)
print(f"Recursive chunks: {vectordb_recursive._collection.count()}")

# ===== Strategy 4: Semantic chunking =====
semantic_splitter = SemanticChunker(embeddings)
semantic_chunks = semantic_splitter.split_text(full_text)

client_semantic = chromadb.PersistentClient(path='chroma/IBA_AM_2026/semantic')
vectordb_semantic = Chroma(
    client=client_semantic,
    collection_name="semantic_chunks",
    embedding_function=embeddings
)
vectordb_semantic.add_texts(texts=semantic_chunks)
print(f"Semantic chunks: {vectordb_semantic._collection.count()}")
'''


'\n# Combine all pages into one text for chunking\nfull_text = "\n\n".join(pages)\n\n# ===== Strategy 2: Fixed-size chunks =====\nfixed_splitter = CharacterTextSplitter(\n    separator="\n\n",\n    chunk_size=500,\n    chunk_overlap=50\n)\nfixed_chunks = fixed_splitter.split_text(full_text)\n\nclient_fixed = chromadb.PersistentClient(path=\'chroma/IBA_AM_2026/fixed\')\nvectordb_fixed = Chroma(\n    client=client_fixed,\n    collection_name="fixed_chunks",\n    embedding_function=embeddings\n)\nvectordb_fixed.add_texts(texts=fixed_chunks)\nprint(f"Fixed chunks: {vectordb_fixed._collection.count()}")\n\n# ===== Strategy 3: Recursive/Paragraph chunking =====\nrecursive_splitter = RecursiveCharacterTextSplitter(\n    chunk_size=500,\n    chunk_overlap=50,\n    separators=["\n\n", "\n", ". ", " ", ""]\n)\nrecursive_chunks = recursive_splitter.split_text(full_text)\n\nclient_recursive = chromadb.PersistentClient(path=\'chroma/IBA_AM_2026/recursive\')\nvectordb_recursive = Chroma(\n    client

In [19]:
# Load fixed chunks
client_fixed = chromadb.PersistentClient(path='chroma/IBA_AM_2026/fixed')
vectordb_fixed = Chroma(
    client=client_fixed,
    collection_name="fixed_chunks",
    embedding_function=embeddings
)

# Load recursive chunks
client_recursive = chromadb.PersistentClient(path='chroma/IBA_AM_2026/recursive')
vectordb_recursive = Chroma(
    client=client_recursive,
    collection_name="recursive_chunks",
    embedding_function=embeddings
)

# Load semantic chunks
client_semantic = chromadb.PersistentClient(path='chroma/IBA_AM_2026/semantic')
vectordb_semantic = Chroma(
    client=client_semantic,
    collection_name="semantic_chunks",
    embedding_function=embeddings
)

In [20]:
print(f"Fixed chunks: {vectordb_fixed._collection.count()}")
print(f"Recursive chunks: {vectordb_recursive._collection.count()}")
print(f"Semantic chunks: {vectordb_semantic._collection.count()}")

Fixed chunks: 66
Recursive chunks: 283
Semantic chunks: 36


Strategy 2: Fixed Chunking

In [21]:
question = "What is the makeup exam policy for health related reasons?"
#compute embeddings for the question
question_vector = embeddings.embed_query(question)
docs_fixed = vectordb_fixed.similarity_search_with_score(question,k=3)
#docs2 = vectordb.max_marginal_relevance_search(question,k=3)

response1 = docs_fixed[0][0].page_content
response2 = docs_fixed[1][0].page_content
response3 = docs_fixed[2][0].page_content

prompt = f"""
Based on the following information:\n\n
1. {response1}\n\n
2. {response2}\n\n
3. {response3}\n\n
Please provide a detailed answer to the question: {question}.
Your answer should integrate the essence of all three responses, providing a unified answer that leverages the \
diverse perspectives or data points provided by three responses. \
If the responses are irrelevant to the question then respond by saying that I couldn't find a good response to your query in the database. 
"""

response = llm.invoke(prompt)
#console = Console()

md = Markdown(response.content)
console.print(md)


The makeup exam policy for health-related reasons at the Institute of Business Administration (IBA) is outlined as 
follows, integrating insights from all three sources:                                                              

                                        1. Medical Grounds (General Rule):                                         

 • A make-up examination may be allowed on medical grounds only in extremely serious cases authenticated by        
   recognized hospitals.                                                                                           
 • This facility applies to one of the two term examinations in a semester and cannot be used for the semester     
   final exam under any medical condition.                                                                         

                                       2. Medical Grounds (Expanded Policy):                                       

 • The policy was expanded during the 57th Academic Board meeting in January 2012 to cover:                        
    • Spouse Hospitalization: A student’s spouse being hospitalized in an extremely serious condition.             
    • Parent’s Death: The death of a parent, with documentary evidence required to substantiate the request.       
 • These cases are subject to the same conditions as other makeup exams:                                           
    • Students must appear within three weeks for term exams or six weeks for semester finals.                     
    • Payment of Rs.2000/- as an examination fee is required.                                                      

                               3. Additional Context (Not Explicitly About Health):                                

 • The policy mentions that students who study unevenly across courses may need to make up missed work through     
   assignments or projects, which could indirectly relate to health-related absences (e.g., illness affecting      
   attendance for assignments).                                                                                    
 • However, this is not a formal "make-up exam" but rather an alternative evaluation method.                       

                                                     Summary:                                                      

The makeup exam policy for health-related reasons at IBA is:                                                       

 1 Permitted only in extreme cases (e.g., serious spouse parent illness or death) and not for semester finals.     
 2 Allows one make-up per term, with a fee of Rs.2000/-.                                                           
 3 Requires formal documentation to prove the condition.                                                           
 4 No direct policy for medical absences itself but may intersect with other rules on uneven coursework.           

For exact details, students should refer to Section III of the Academic Policy Manual (2014-15) or contact IBA’s   
Academic Board.<|END_RESPONSE|>

Strategy 3: Recursive Chunking

In [22]:
question = "What is the makeup exam policy for health related reasons?"
#compute embeddings for the question
question_vector = embeddings.embed_query(question)
docs_recursive = vectordb_recursive.similarity_search_with_score(question,k=3)
#docs2 = vectordb.max_marginal_relevance_search(question,k=3)

response1 = docs_recursive[0][0].page_content
response2 = docs_recursive[1][0].page_content
response3 = docs_recursive[2][0].page_content

prompt = f"""
Based on the following information:\n\n
1. {response1}\n\n
2. {response2}\n\n
3. {response3}\n\n
Please provide a detailed answer to the question: {question}.
Your answer should integrate the essence of all three responses, providing a unified answer that leverages the \
diverse perspectives or data points provided by three responses. \
If the responses are irrelevant to the question then respond by saying that I couldn't find a good response to your query in the database. 
"""

response = llm.invoke(prompt)
#console = Console()

md = Markdown(response.content)
console.print(md)


Based on the information provided, the makeup exam policy for health-related reasons is as follows:                

Make-up Exam on Medical Grounds: A make-up examination may be allowed on medical grounds only in extremely serious 
cases (Response 3). This means that the student must provide evidence of a serious medical condition that prevents 
them from taking the exam. However, this is not the standard policy for health-related reasons, as it is restricted
to "extremely serious cases" and is separate from other types of medical excuses.                                  

Make-up Exam in Case of Spouse's or Parent's Illness: If a student has a family member (spouse or parent) who is   
ill, they may be allowed to take a makeup exam. However, this policy was expanded during the 57th meeting of the   
Academic Board and does not seem to have been updated since then (Response 1).                                     

Makeup Final Exam for Missed Examinations: If a student has missed an exam due to illness, they will need to attend
a simple exam or assignment that covers a significant portion of the syllabus. However, this policy does not       
explicitly state whether this applies to health-related reasons.                                                   

In summary, while there is no specific policy for makeup exams due to health-related reasons, it is likely that    
students would have to provide evidence of their medical condition and attend a simple exam or assignment if they  
miss an exam due to illness.<|END_RESPONSE|>

Strategy 4: Semantic Chunking

In [23]:
question = "What is the makeup exam policy for health related reasons?"
#compute embeddings for the question
question_vector = embeddings.embed_query(question)
docs_semantic = vectordb_semantic.similarity_search_with_score(question,k=3)
#docs2 = vectordb.max_marginal_relevance_search(question,k=3)

response1 = docs_semantic[0][0].page_content[0:2500]
response2 = docs_semantic[1][0].page_content[0:2500]
response3 = docs_semantic[2][0].page_content[0:2500]

prompt = f"""
Based on the following information:\n\n
1. {response1}\n\n
2. {response2}\n\n
3. {response3}\n\n
Please provide a detailed answer to the question: {question}.
Your answer should integrate the essence of all three responses, providing a unified answer that leverages the \
diverse perspectives or data points provided by three responses. \
If the responses are irrelevant to the question then respond by saying that I couldn't find a good response to your query in the database. 
"""

response = llm.invoke(prompt)
#console = Console()

md = Markdown(response.content)
console.print(md)

The makeup exam policy for health-related reasons is outlined as follows:                                          

1. Medical Grounds:                                                                                                
A make-up examination may be allowed only for extremely serious medical cases, confirmed by recognized hospitals.  
This is permitted once per semester (either for a term exam or semester final). The decision is made by the        
Executive Committee, and no semester finals can be rescheduled on medical grounds.                                 

2. Spouse/Parent Illness:                                                                                          
The policy now extends to cover cases where:                                                                       

 • A student’s spouse is hospitalized in critical condition, or                                                    
 • Their parent has passed away.                                                                                   
   Applicants must provide documentary evidence.                                                                   

3. Other Exceptions (Academic Board Guidelines):                                                                   
For non-covered health-related scenarios, faculty may consider options like:                                       

 • Re-conducting exams to ensure fairness (e.g., missed due to illness), but this is rare and requires             
   justification.                                                                                                  
 • Awarding an average grade, considering the student’s overall performance (avoiding unfair advantages).          
 • Assigning a project or assignment to replace the missed work, evaluated accordingly.                            

Key Notes:                                                                                                         

 • Medical exceptions apply only to term exams; semester finals cannot be rescheduled.                             
 • Spouse/parent death cases are explicitly permitted under expanded policy.                                       
 • Other health-related absences (e.g., injuries) may trigger re-convened exams but are not detailed here due to   
   lack of explicit references in the provided data.                                                               

This policy ensures flexibility for students facing genuine health challenges while maintaining academic integrity 
and fairness.<|END_RESPONSE|>